
# ColliderML platform tutorial

A six-chapter walkthrough of the ColliderML platform from the
perspective of a researcher who's never seen it before. At the end
you'll have:

- **downloaded** a real tracker-hit dataset from HuggingFace,
- **simulated** a small sample locally via the official container,
- **submitted** a simulation job to the SaaS backend,
- **scored** a tracking-algorithm submission against the leaderboard,
- **published** your model as an HF repo that the model-zoo Space
  picks up.

Each chapter ends with a **"what just happened"** explainer that
unpacks the architecture under the hood, so the tutorial doubles as
a conceptual map of the platform.

## Prerequisites

| | |
|---|---|
| **Python** | 3.10 or newer. |
| **Install** | `pip install --pre 'colliderml[all]'` (pre-release until `0.4.0` final). |
| **Disk headroom** | ~2 GB for the tracker-hits slice we download in Chapter 1. |
| **For Chapter 2** (local simulation): Docker or Podman on your `$PATH`, plus ~12 GB for the pipeline container + Geant4 datasets. |
| **For Chapters 3 and 4** (SimaaS and leaderboard): a running backend. Run `scripts/setup_tutorial_env.sh path/to/colliderml-production/backend` once. |
| **HuggingFace account + token** for Chapters 3–5. `huggingface-cli login` gets it into place. |

Every code cell below degrades gracefully if its prerequisites aren't
present (prints a message explaining what's missing and moves on), so
you can skim the notebook start-to-finish even without the full setup.

## How the pieces fit together

```
┌──────────────┐   HTTP    ┌────────────────┐   SFAPI   ┌───────────┐
│ your laptop  │──────────▶│ backend (FastAPI│──────────▶│ Perlmutter│
│ colliderml.* │           │  +  Postgres)  │            │  (Slurm)  │
└──────┬───────┘           └────────┬───────┘            └─────┬─────┘
       │                            │                          │
       │  HuggingFace Hub           │   uploads artefacts       │
       ▼                            ▼                          ▼
┌──────────────┐            ┌──────────────────┐      ┌───────────┐
│ datasets,    │            │ HF dataset repos │◀─────│ pipeline  │
│ model zoo,   │            │  (simulated      │      │  output   │
│ Spaces       │            │   events)        │      └───────────┘
└──────────────┘            └──────────────────┘
```

Chapters 1 and 5 use only the left column (HF Hub). Chapter 2 uses
your laptop. Chapters 3 and 4 use the middle and right columns.


In [ ]:
# Environment probe — every chapter below keys off these.
import os, shutil, socket

import colliderml

HAS_BACKEND = False
BACKEND_URL = os.environ.get("COLLIDERML_BACKEND", "http://localhost:8000")
try:
    import urllib.request
    with urllib.request.urlopen(f"{BACKEND_URL}/healthz", timeout=2) as r:
        HAS_BACKEND = r.status == 200
except Exception:
    HAS_BACKEND = False

HAS_DOCKER = shutil.which("docker") is not None or shutil.which("podman") is not None

try:
    from huggingface_hub import HfApi
    HF_USER = HfApi().whoami().get("name", None)
except Exception:
    HF_USER = None

print(f"colliderml:   {colliderml.__version__}")
print(f"backend URL:  {BACKEND_URL} ({'reachable' if HAS_BACKEND else 'NOT reachable — chapters 3, 4 will skip'})")
print(f"docker/podman: {'found' if HAS_DOCKER else 'NOT found — chapter 2 will skip'}")
print(f"HF user:      {HF_USER or 'not logged in — chapter 5 will skip'}")


## Chapter 1 — Loading data from HuggingFace

The canonical ColliderML datasets live at
[`CERN/ColliderML-Release-1`](https://huggingface.co/datasets/CERN/ColliderML-Release-1).
Each dataset is named `<channel>_<pileup>`, e.g. `ttbar_pu0` or
`higgs_portal_pu200`. The library's `load()` handles cache-aware
download + Polars loading in one call.

We'll start by discovering what's available and then pulling a small
slice of `ttbar_pu0`.


In [ ]:
from colliderml.core.hf_download import discover_remote_configs

configs = discover_remote_configs("CERN/ColliderML-Release-1")
print(f"{len(configs)} configs available. A handful:")
for c in configs[:6]:
    print(f"  - {c}")

In [ ]:
# Load 200 events from the ttbar_pu0 tracker-hits slice. First run
# downloads the parquet shards (tens of MB); subsequent runs are cached.
frames = colliderml.load(
    "ttbar_pu0",
    tables=["tracker_hits", "particles"],
    max_events=200,
)
for name, frame in frames.items():
    print(f"{name}: {len(frame)} rows, columns {frame.columns[:5]}...")

In [ ]:
# Peek inside one event's tracker-hits row. Note the list-per-event
# schema: each row is one event, each column is a list across hits.
row = frames["tracker_hits"].row(0, named=True)
print(f"event_id:        {row['event_id']}")
print(f"number of hits:  {len(row['hit_id'])}")
print(f"first few pids:  {row['particle_id'][:5]}")


### What just happened

1. `discover_remote_configs` hits the HF dataset repo and extracts unique
   `data/<config>/...` prefixes — the config list is derived from what's
   on disk in the repo, not a hardcoded manifest.
2. `colliderml.load(...)` resolved the name `"ttbar_pu0"` into the set
   of parquet shards for the requested tables, downloaded any missing
   shards into `$COLLIDERML_DATA_DIR` (defaults to `~/.cache/colliderml`),
   and loaded them with the Polars-backed loader.
3. `max_events=200` is an in-memory slice — the entire shard has to be
   downloaded once, then we take the first 200 rows. For a tiny sample
   this is wasteful; for real workflows the cache pays for itself from
   the second call onward.

The nested schema (one row per event, columns as lists across hits)
is central to the library's performance story — Polars can lazily
scan these shards without exploding them into flat tables.



## Chapter 2 — Run the pipeline yourself: local simulation

ColliderML ships a container
(`ghcr.io/opendatadetector/sw:0.2.2_linux-ubuntu24.04_gcc-13.3.0`)
that bundles MadGraph, Pythia, Geant4+ddsim, and ACTS. The
`colliderml.simulate` subpackage drives the full pipeline — hard
scatter → showered events → detector simulation → track reconstruction
— in a single call.

For this chapter we use the `single-muon` preset (fastest, ~5 min).


In [ ]:
# List the bundled presets. Pick by speed (single-muon fastest, then
# *-quick, then *-dev, then the full *-benchmark workloads).
from colliderml.simulate import load_presets

presets = load_presets()
for name, cfg in presets.items():
    print(f"  {name:24}  channel={cfg['channel']:14} events={cfg['events']:>4} pileup={cfg['pileup']}")

In [ ]:
# Actually run the pipeline. Skipped when no container runtime is on
# the host — set HAS_DOCKER at the top of this notebook.
if HAS_DOCKER:
    result = colliderml.simulate(preset="single-muon", quiet=True)
    print(f"run directory:  {result.run_dir}")
    print(f"stages run:     {[s.name for s in result.stages]}")
    print(f"output files:   {sorted(p.name for p in result.run_dir.iterdir())[:6]}")
else:
    print("Chapter 2 skipped — no Docker/Podman runtime on this host.")
    print("On a laptop or dev box with either installed, this call runs")
    print("the full pipeline (~5 minutes for single-muon) and writes to")
    print("./output/runs/0/.")


### What just happened

On first call, `colliderml.simulate` did three things before running
anything:

1. **Detected the runtime** — `docker` preferred, `podman` fallback.
2. **Cloned `colliderml-production@pipeline-v0.1.0`** into `.cache/`
   for the pipeline scripts. This auto-clone pattern mirrors the one
   already used for the ODD geometry and the MG5 ↔ Pythia8 interface.
3. **Built the ODD detector geometry and downloaded Geant4 datasets**
   into the cache (both one-time, ~10 minutes combined).

Then for each stage (Pythia → DDSim → Digi+Reco → Parquet) it started
a container with the pipeline script for that stage, mounted the run
directory, and ran to completion. The `SimulationResult` you got back
records which stages ran, how long each took, and where the outputs
are.

The **same** parquet output format that Chapter 1 downloaded from HF
is what this pipeline writes locally — so you can round-trip a
simulated sample through `colliderml.load()` without caring about the
source.



## Chapter 3 — Scale up: Simulation as a Service

Single-muon runs in 5 minutes on your laptop. A full
`ttbar + pileup 200` run takes a day of CPU time and tens of GB of
output — not something you want to babysit locally. The SaaS backend
accepts a JSON request, queues a Slurm job on NERSC Perlmutter via
the SFAPI, polls it to completion, and uploads the artefacts to an
HF dataset repo under your account.

From the user's side it's three calls: `submit`, `wait_for`, then
download the resulting HF repo like any other dataset.


In [ ]:
from colliderml import remote

if not HAS_BACKEND:
    print("Chapter 3 skipped — no backend reachable at", BACKEND_URL)
    print("To run it locally: `scripts/setup_tutorial_env.sh path/to/colliderml-production/backend`")
    print("Then restart the notebook so the environment probe picks it up.")
else:
    print(f"You have {remote.balance():.0f} credits available.")

In [ ]:
# Submit a tiny simulation — 10 events of ttbar + PU=10. In mock-SFAPI
# mode the job completes in ~2 seconds; with real SFAPI credentials set
# on the backend, this actually submits an sbatch script to Perlmutter.
if HAS_BACKEND:
    sub = remote.submit(channel="ttbar", events=10, pileup=10)
    print(f"request_id:  {sub.request_id}")
    print(f"state:       {sub.state}")
    print(f"credits:     {sub.credits_charged}")

In [ ]:
# Poll to completion.
if HAS_BACKEND:
    final = remote.wait_for(sub.request_id, poll_interval=1, timeout=120)
    print(f"final state:  {final.state}")
    print(f"output repo:  {final.output_hf_repo}")

In [ ]:
# Submitting the same request again hits the backend's dedup cache
# (same channel/events/pileup/seed hash, same output) — zero credits
# charged the second time.
if HAS_BACKEND:
    dup = remote.submit(channel="ttbar", events=10, pileup=10)
    print(f"request_id:      {dup.request_id}")
    print(f"cached:          {dup.request_id == sub.request_id}")
    print(f"credits charged: {dup.credits_charged}")


### What just happened

Inside the backend, `POST /v1/simulate` did the following:

1. Hashed the request config and checked the dedup table — if a
   completed request with the same hash exists within the last
   7 days, returns it verbatim (credit_charged=0).
2. Ran credit / abuse checks. The kill switch (`/admin/freeze`)
   short-circuits here with a 503 if an operator has tripped it.
3. Inserted a row into the `simulation_requests` table with state
   `queued`.
4. Handed off to `SFAPIRunner.submit`:
   - **With real credentials** (`SFAPI_CLIENT_ID/SECRET` set): renders
     `app/sbatch_template.sh.j2`, uploads it to `/pscratch/sd/.../colliderml/`,
     POSTs to the SFAPI `/compute/jobs` endpoint, spawns a polling
     task that hits `/compute/jobs/{id}` every 60 s.
   - **In mock mode** (default for local dev): spawns a task that
     marks the request completed after 2 seconds.
5. Returned the `(request_id, state=queued, credits_charged, …)`
   tuple.

`remote.wait_for` polls `GET /v1/requests/{id}` every
`poll_interval` seconds until the state is terminal
(`completed`/`failed`/`cancelled`). On success you get back an
`output_hf_repo` URL — a fresh HF dataset repo under your account
containing the parquet shards, loadable via the same
`colliderml.load()` you used in Chapter 1.



## Chapter 4 — Benchmark a tracking algorithm

The `tracking` task asks you to reconstruct particle tracks from
detector hits in `ttbar_pu200` events. The dataset ships *candidate*
tracker hits; your job is to group them into track assignments. A
submission is a parquet file with three columns: `event_id`,
`hit_id`, `track_id`.

### The tracking efficiency definition

The primary metric is **TrackML weighted efficiency**, scored by the
double-majority rule on a held-out eval split (events 90 000–99 999):

- A reconstructed track is **correct** if **one truth particle owns
  ≥50% of its hits** AND **that particle contributes ≥50% of its own
  hits to the track**.
- Efficiency = sum of hit weights in correct tracks ÷ total hit
  weight.

Two companion metrics unpack failure modes:

- **Fake rate** — fraction of tracks failing the first half of the
  double-majority rule (no single particle dominates).
- **Duplicate rate** — fraction of truth particles matched to more
  than one reconstructed track.
- **Physics efficiency (pT > 1 GeV)** — fraction of high-pT primary
  particles with any reconstructed track, a physics-centric
  complement to the TrackML metric.

All four are scored server-side on truth that's never shipped to the
client.

### A pedagogical submission

The library ships two oracle baselines under
`colliderml.tasks.tracking.baselines.oracle`:

- `perfect_oracle_predictions(truth_hits)` — assigns each hit to its
  truth particle. Scores ~1.0 on everything. Upper bound.
- `noised_oracle_predictions(truth_hits, *, split_fraction=...,
  merge_fraction=...)` — starts from the perfect oracle and
  deliberately degrades it. `split_fraction` breaks tracks into two
  asymmetric pieces (attacks the "track owns majority of particle's
  hits" half of the rule); `merge_fraction` unifies track pairs
  within an event (attacks the "one particle owns majority of
  track's hits" half).

Let's evaluate a noised-oracle submission locally, then submit it to
the backend.


In [ ]:
import colliderml

# Load the eval split's truth hits directly via the task API.
tracking = colliderml.tasks.get("tracking")
# Use a small subrange for the demo (the full eval split is 10 000 events).
small_eval_range = (tracking.eval_event_range[0], tracking.eval_event_range[0] + 50)
truth = tracking.load(tables=["tracker_hits"], event_range=small_eval_range)["tracker_hits"]
print(f"truth hits in demo range: {truth.num_rows:,}")

In [ ]:
# Build a noised-oracle submission. 10% splits + 10% merges.
from colliderml.tasks.tracking.baselines import noised_oracle_predictions, perfect_oracle_predictions

perfect_preds = perfect_oracle_predictions(truth)
noised_preds = noised_oracle_predictions(truth, split_fraction=0.1, merge_fraction=0.1, seed=42)

# Score both locally against the demo slice.
def _score_local(preds):
    from colliderml.tasks.tracking.metrics import (
        duplicate_rate,
        fake_rate,
        trackml_weighted_efficiency,
    )
    return {
        "trackml_eff": trackml_weighted_efficiency(preds, truth),
        "fake_rate":   fake_rate(preds, truth),
        "dup_rate":    duplicate_rate(preds, truth),
    }

print("perfect oracle:", _score_local(perfect_preds))
print("noised oracle: ", _score_local(noised_preds))

In [ ]:
# Submit to the leaderboard. Backend re-scores on the full eval split
# (events 90 000-99 999) using truth that's never shipped to clients.
if HAS_BACKEND:
    import pyarrow.parquet as pq, io
    buf = io.BytesIO()
    pq.write_table(noised_preds, buf)
    buf.seek(0)
    # NOTE: the real API is colliderml.tasks.submit("tracking", predictions_table).
    # Here we skip over a tiny wrapper that does the parquet serialization.
    result = colliderml.tasks.submit("tracking", noised_preds)
    print(f"scores:          {result['scores']}")
    print(f"credits earned:  {result['credits_earned']}")
else:
    print("Chapter 4 leaderboard submit skipped — no backend reachable.")
    print("The local evaluation above is the same code the backend runs;")
    print("the only difference is the eval range (50 events here vs 10 000")
    print("on the server) and the fact that the server awards credits for a new best.")


### What just happened

The backend's `POST /v1/benchmark/tracking/submit` took the parquet
bytes, loaded them with pyarrow, ran the same
`trackml_weighted_efficiency` / `fake_rate` / `duplicate_rate` code
you just ran locally — but on the full 10 000-event eval split
backed by a truth table that never leaves the server. If the result
beats the current best on any metric with `higher_is_better=True`, a
credits reward is written to the user's ledger. The leaderboard
Space (`spaces/leaderboard/`) polls `/v1/leaderboard/tracking` to
render the table you'd see at <https://huggingface.co/spaces/OpenDataDetector/colliderml-leaderboard>.

**Why an oracle baseline is a useful submission.** It's not a
tracking algorithm — it's a calibration point. Perfect oracle
scores ≈1 by construction, so it tells you the metric's true
ceiling. Noised oracle with small fractions shows how quickly the
TrackML score punishes each failure mode: a single `split_fraction`
knob that generates track fragments at ~30% of their parent's hit
count drops the efficiency from 1.0 roughly linearly in
`split_fraction`, because each split turns one correct track into
(one correct track + one fake). Mergers are harsher: a single merge
sacrifices **both** contributing tracks.

When you write a real tracker (DBSCAN on (η, φ, r), a GNN, a CKF),
this is the yardstick.



## Chapter 5 — Share your model: the model zoo

The ColliderML model zoo is a thin HuggingFace Hub filter: any model
on the Hub tagged `colliderml` shows up in
[`spaces/model-zoo`](https://huggingface.co/spaces/OpenDataDetector/colliderml-model-zoo).
There's no backend registry, no approval process — the `tag` is the
contract. Pushing a model is a standard `huggingface_hub.create_repo`
+ `upload_folder`.

We'll wrap our toy noised-oracle submission as a "model" for
completeness.


In [ ]:
import pathlib, textwrap

if not HF_USER:
    print("Chapter 5 skipped — no HF user (try `huggingface-cli login`).")
else:
    model_dir = pathlib.Path("/tmp/colliderml-tutorial-model")
    model_dir.mkdir(exist_ok=True)
    # A stub "model". Real submissions would be a torch state dict,
    # an ONNX file, a pickled scikit pipeline, etc.
    (model_dir / "config.json").write_text(
        '{"kind": "noised-oracle", "split_fraction": 0.1, "merge_fraction": 0.1, "seed": 42}\n'
    )
    (model_dir / "README.md").write_text(textwrap.dedent('''        ---
        tags:
          - colliderml
          - tracking
        license: mit
        ---
        # Noised-oracle tracker (tutorial)

        Pedagogical baseline for the ColliderML tracking task.
        Scores ~0.8 TrackML weighted efficiency at
        `split_fraction=0.1, merge_fraction=0.1`. Not a real tracker —
        serves as a calibration point for evaluating your own
        submissions.
    '''))
    print(f"Model dir prepared at: {model_dir}")
    for f in sorted(model_dir.iterdir()):
        print(f"  {f.name:16} {f.stat().st_size} bytes")

In [ ]:
# Push. Skipped without an HF user.
if HF_USER:
    from huggingface_hub import create_repo, upload_folder
    repo_id = f"{HF_USER}/colliderml-tutorial-tracker"
    create_repo(repo_id, exist_ok=True)
    upload_folder(folder_path=str(model_dir), repo_id=repo_id)
    print(f"published: https://huggingface.co/{repo_id}")
    print("Refresh the model-zoo Space — your repo should now appear.")


### What just happened

You created an HF model repo under your namespace with a README that
has `tags: [colliderml, tracking]` in its YAML frontmatter. The
model-zoo Space runs
```python
huggingface_hub.list_models(filter="colliderml")
```
on every page load and renders whatever comes back. Your repo is now
one of the results.

**Tradeoff.** The simplicity is deliberate: no backend registry
means no moderation, no provenance linking, no guaranteed
compatibility with evaluation tooling. The contract is just the tag.
A future iteration could add a `/v1/models` backend endpoint that
stitches together a model card with its leaderboard score(s), or
runs the model's inference against the eval set to produce a
deterministic score — but that's a later conversation.



## Chapter 6 — Recap and next steps

You just drove every public surface of the ColliderML platform:

| Chapter | Surface |
|---|---|
| 1 | `colliderml.load()`, `colliderml.core.hf_download` |
| 2 | `colliderml.simulate()`, preset catalogue, container auto-clone |
| 3 | `colliderml.remote.{submit, wait_for, balance}`, dedup cache |
| 4 | `colliderml.tasks.{get, evaluate, submit}`, tracking metrics, oracle baselines |
| 5 | `huggingface_hub.create_repo`, the `tags: [colliderml]` contract |

### Reference card

```python
import colliderml
from colliderml import remote
from colliderml.tasks.tracking.baselines import noised_oracle_predictions

# 1. Load data
frames = colliderml.load("ttbar_pu0", tables=["tracker_hits"], max_events=200)

# 2. Run the pipeline locally
result = colliderml.simulate(preset="single-muon")

# 3. Submit a remote simulation
sub = remote.submit(channel="ttbar", events=10_000, pileup=200)
final = remote.wait_for(sub.request_id)

# 4. Score a tracking submission
preds = noised_oracle_predictions(truth_hits, split_fraction=0.05)
result = colliderml.tasks.submit("tracking", preds)

# 5. Publish a model (standard HF)
from huggingface_hub import create_repo, upload_folder
create_repo(f"{user}/my-tracker", exist_ok=True)
upload_folder(folder_path="...", repo_id=f"{user}/my-tracker")
```

### What's not covered

- **Other tasks.** `jets`, `anomaly`, `tracking_latency`,
  `tracking_small`, `data_loading` all ship with reference baselines
  and the same `colliderml.tasks.{get, evaluate, submit}` surface.
- **Writing your own task.** Subclass
  `colliderml.tasks.BenchmarkTask`, register with `@register`, ship
  reference baselines. The admin onboards the task by adding it to
  the backend's task list.
- **Running the platform yourself.** Operator docs live at
  `docs/internal/*` on the public docs site (direct URL, not in the
  sidebar). Covers backend deployment, SFAPI credentials, admin
  Space operations, and the container image build checklist.
- **Webhooks + multi-node.** The backend supports POSTing a webhook
  when a request completes and has multi-node sbatch templates for
  the `*-benchmark` presets. Useful for CI/CD-style workflows.

Issues, questions, ideas: `github.com/OpenDataDetector/ColliderML/issues`.
